# Praktikum 2: Klasifikasi Multikelas dengan Dataset CIFAR-10 Menggunakan CNN
**Mata Kuliah:** Pembelajaran Mesin (Machine Learning)  
**Jobsheet:** JS14 – Convolutional Neural Network  
**Topik:** Klasifikasi 10 Objek Citra CIFAR-10 Menggunakan TensorFlow & Keras  

---
### Deskripsi
Praktikum ini merujuk pada tutorial resmi TensorFlow: [TensorFlow CNN Image Classification](https://www.tensorflow.org/tutorials/images/cnn).

Tujuan praktikum adalah membangun model Convolutional Neural Network (CNN) untuk mengenali dan mengklasifikasikan 10 kelas objek yang terdapat pada **Dataset CIFAR-10**.
- **Karakteristik Dataset CIFAR-10:**
  - Terdiri dari 60.000 citra berwarna RGB dengan dimensi $32 \times 32$ piksel.
  - Memiliki 10 kelas yang seimbang (masing-masing 6.000 citra):
    - 0: `airplane` (pesawat terbang)
    - 1: `automobile` (mobil)
    - 2: `bird` (burung)
    - 3: `cat` (kucing)
    - 4: `deer` (rusa)
    - 5: `dog` (anjing)
    - 6: `frog` (katak)
    - 7: `horse` (kuda)
    - 8: `ship` (kapal)
    - 9: `truck` (truk)
  - Pembagian: 50.000 citra untuk data latih (*training*) dan 10.000 citra untuk data uji (*testing*).


## Langkah 1: Load Library
Sesuai modul:
- Mengimpor pustaka TensorFlow, sub-modul datasets, layers, models, serta matplotlib.pyplot.


In [1]:
# Langkah 1 - Load Library
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

print(f"TensorFlow Version: {tf.__version__}")


TensorFlow Version: 2.16.1


## Langkah 2: Unduh Dataset CIFAR-10 dan Normalisasi
Dataset diunduh langsung melalui `datasets.cifar10.load_data()`, kemudian nilai intensitas piksel citra [0, 255] dinormalisasi ke rentang [0.0, 1.0].


In [2]:
# Langkah 2 - Unduh Dataset CIFAR
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

print(f"Bentuk train_images: {train_images.shape}, train_labels: {train_labels.shape}")
print(f"Bentuk test_images : {test_images.shape}, test_labels : {test_labels.shape}")


Bentuk train_images: (50000, 32, 32, 3), train_labels: (50000, 1)
Bentuk test_images : (10000, 32, 32, 3), test_labels : (10000, 1)


## Langkah 3: Verifikasi Data
Menampilkan 25 sampel gambar pertama dari dataset training beserta nama kelasnya untuk memastikan data telah termuat dengan benar.


In [3]:
# Langkah 3 - Verifikasi Data
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    # The CIFAR labels happen to be arrays, which is why you need the extra index
    plt.xlabel(class_names[train_labels[i][0]])
plt.tight_layout()
plt.show()


## Langkah 4: Pembuatan Arsitektur Model CNN
Arsitektur model dibangun secara bertahap:
- **Langkah 4.1:** Membuat blok lapisan konvolusi dan pooling:
  - `Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3))`
  - `MaxPooling2D((2, 2))`
  - `Conv2D(64, (3, 3), activation='relu')`
  - `MaxPooling2D((2, 2))`
  - `Conv2D(64, (3, 3), activation='relu')`
- **Langkah 4.2:** Memeriksa arsitektur konvolusi (`model.summary()`).
- **Langkah 4.3:** Menambahkan lapisan *Fully Connected*:
  - `Flatten()`
  - `Dense(64, activation='relu')`
  - `Dense(10)` (output logits untuk 10 kelas)
- **Langkah 4.4:** Memeriksa arsitektur lengkap model.
- **Langkah 4.5:** Mengompilasi model dengan optimizer `adam`, loss `SparseCategoricalCrossentropy(from_logits=True)`, dan metrik `accuracy`.


In [4]:
# Langkah 4.1 - Buat Layer Konvolusi
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Langkah 4.2 - Cek Arsitektur Konvolusi
print("--- Arsitektur Blok Konvolusi ---")
model.summary()


--- Arsitektur Blok Konvolusi ---
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 30, 30, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 15, 15, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 13, 13, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 6, 6, 64)          0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 4, 4, 64)          36928     
                                                                 
Total params: 56320 (2

In [5]:
# Langkah 4.3 - Tambahkan Layer Fully Connected
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10))

# Langkah 4.4 - Cek Arsitektur Model CNN Lengkap
print("--- Arsitektur Model CNN Lengkap ---")
model.summary()

# Langkah 4.5 - Compile Model CNN
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


--- Arsitektur Model CNN Lengkap ---
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 30, 30, 32)        896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 15, 15, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 13, 13, 64)        18496     
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 6, 6, 64)          0         
 g2D)                                                            
                                                                 
 conv2d_2 (Conv2D)           (None, 4, 4, 64)          36928     
                                                                 
 flatten (Flatten) 

## Langkah 5: Fit Model (Training)
Melatih model selama 10 epoch dengan data validasi dari `test_images` dan `test_labels`.


In [6]:
# Langkah 5 - Fit Model
history = model.fit(
    train_images, 
    train_labels, 
    epochs=10, 
    validation_data=(test_images, test_labels)
)


Epoch 1/10
1563/1563 [==============================] - 42s 26ms/step - loss: 1.5142 - accuracy: 0.4485 - val_loss: 1.2584 - val_accuracy: 0.5510
Epoch 2/10
1563/1563 [==============================] - 41s 26ms/step - loss: 1.1524 - accuracy: 0.5910 - val_loss: 1.0542 - val_accuracy: 0.6275
Epoch 3/10
1563/1563 [==============================] - 41s 26ms/step - loss: 1.0028 - accuracy: 0.6482 - val_loss: 0.9854 - val_accuracy: 0.6542
Epoch 4/10
1563/1563 [==============================] - 41s 26ms/step - loss: 0.9084 - accuracy: 0.6815 - val_loss: 0.9248 - val_accuracy: 0.6784
Epoch 5/10
1563/1563 [==============================] - 41s 26ms/step - loss: 0.8354 - accuracy: 0.7064 - val_loss: 0.9015 - val_accuracy: 0.6890
Epoch 6/10
1563/1563 [==============================] - 41s 26ms/step - loss: 0.7762 - accuracy: 0.7285 - val_loss: 0.8752 - val_accuracy: 0.6985
Epoch 7/10
1563/1563 [==============================] - 41s 26ms/step - loss: 0.7251 - accuracy: 0.7454 - val_loss: 0.8524 -

## Langkah 6: Evaluasi Model dan Visualisasi Kurva Akurasi
Memvisualisasikan grafik akurasi training (`accuracy`) dan akurasi validasi (`val_accuracy`) pada setiap epoch.


In [7]:
# Langkah 6 - Evaluasi Model
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.4, 1])
plt.title('Kurva Akurasi Pelatihan vs Validasi (CIFAR-10 Baseline)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)


313/313 - 2s - loss: 0.8712 - accuracy: 0.7105 - 2s/epoch - 7ms/step


## Langkah 7: Cetak Hasil Akurasi
Menjawab pertanyaan modul: ***"Berapa akurasi yang Anda dapatkan?"***


In [8]:
# Langkah 7 - Cetak Hasil Akurasi
print(f"Akurasi Data Uji (Test Accuracy): {test_acc:.4f} ({test_acc * 100:.2f}%)")
print(f"Loss Data Uji    (Test Loss)    : {test_loss:.4f}")


Akurasi Data Uji (Test Accuracy): 0.7105 (71.05%)
Loss Data Uji    (Test Loss)    : 0.8712


### Pembahasan dan Analisis Praktikum 2:

1. **Akurasi yang Diperoleh:**  
   Model CNN sederhana ini berhasil memperoleh akurasi data uji sebesar **71.05%** (dengan loss 0.8712) setelah 10 epoch pelatihan.

2. **Analisis Overfitting pada Model Sederhana:**  
   - Akurasi data latih meningkat hingga **78.94%**, sementara akurasi data validasi mulai melambat/stagnan di kisaran **70%–71%** sejak epoch ke-7.
   - Hal ini mengindikasikan bahwa model baseline mulai mengalami gejala **overfitting** ringan karena belum dilengkapi teknik regularisasi seperti *Dropout*, *Batch Normalization*, maupun *Data Augmentation*.
   - Temuan ini menjadi dasar penting untuk melakukan modifikasi arsitektur pada **Tugas Praktikum JS14** guna menembus batas akurasi testing **> 80%**.
